In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install anthropic IPython

In [ ]:
import base64
from anthropic import Anthropic
from IPython.display import Image
from openai import OpenAI
from google import genai
from google.genai import types
import os
import pandas as pd
from pathlib import Path
import json
from google.colab import userdata

# API
OpenAI_API_KEY = "sk-proj-"
Gemini_API_KEY = "AI"
Claude_API_KEY = "sk-ant-api03--"

openai = OpenAI(api_key=OpenAI_API_KEY)
claude = Anthropic(api_key=Claude_API_KEY)
gemini = genai.Client(api_key=Gemini_API_KEY)

def get_base64_encoded_image(image_path):
    with open(image_path, "rb") as image_file:
        binary_data = image_file.read()
        base_64_encoded_data = base64.b64encode(binary_data)
        base64_string = base_64_encoded_data.decode('utf-8')
        return base64_string


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


In [ ]:
import pandas as pd
from pathlib import Path

# Excel file containing meta data
in_path = "/content/drive/MyDrive/spirituality/images/Instagram_images_connectedness_type_goldstandard.xlsx"
df = pd.read_excel(in_path)

cols = {c.lower(): c for c in df.columns}
filename_col = cols.get("filename") or cols.get("file") or cols.get("image") or cols.get("image_name")
tag_col = cols.get("tag") or cols.get("label") or cols.get("category")

if filename_col is None or tag_col is None:
    raise ValueError(f"Could not find required columns. Available columns: {list(df.columns)}")


# /content/drive/MyDrive/spirituality/images/{tag}/{Filename}
def build_path(row):
    fname = str(row[filename_col]).strip()
    tag = str(row[tag_col]).strip()
    return f"/content/drive/MyDrive/spirituality/images/{tag}/{fname}"

df["path"] = df.apply(build_path, axis=1)

out_path_with_paths = "/content/Instagram_images_connectedness_type_goldstandard_with_paths.xlsx"
df.to_excel(out_path_with_paths, index=False)

In [ ]:
df

,Filename,URL,tag,Whether it is spirituality (1 means yes),Connectedness Type,path
0,1.jpg,https://scontent-zrh1-1.cdninstagram.com/v/t51...,hindu,1,transcendence,/content/drive/MyDrive/spirituality/images/hin...
1,2.jpg,https://scontent-zrh1-1.cdninstagram.com/v/t51...,hindu,1,nature,/content/drive/MyDrive/spirituality/images/hin...
2,4.jpg,https://scontent-zrh1-1.cdninstagram.com/v/t51...,hindu,1,transcendence,/content/drive/MyDrive/spirituality/images/hin...
3,30.jpg,https://scontent-zrh1-1.cdninstagram.com/v/t51...,hindu,1,transcendence,/content/drive/MyDrive/spirituality/images/hin...
4,31.jpg,https://scontent-zrh1-1.cdninstagram.com/v/t51...,hindu,1,transcendence,/content/drive/MyDrive/spirituality/images/hin...
...,...,...,...,...,...,...
188,45.jpg,https://scontent-zrh1-1.cdninstagram.com/v/t51...,openchristian,1,Transcendence,/content/drive/MyDrive/spirituality/images/ope...
189,300.jpg,https://scontent-zrh1-1.cdninstagram.com/v/t39...,confucianism,1,nature,/content/drive/MyDrive/spirituality/images/con...
190,308.jpg,https://scontent-zrh1-1.cdninstagram.com/v/t51...,confucianism,1,self,/content/drive/MyDrive/spirituality/images/con...
191,315.jpg,https://scontent-zrh1-1.cdninstagram.com/v/t51...,confucianism,1,nature,/content/drive/MyDrive/spirituality/images/con...


In [ ]:
import os
import base64
import pandas as pd
from pathlib import Path
import json
from google.colab import userdata
from anthropic import Anthropic
from openai import OpenAI
from google import genai
from google.genai import types

# Settings
OUTPUT_XLSX = "/content/drive/MyDrive/spirituality/Instagram_images_connectedness_type_goldstandard_with_paths_annotated.xlsx"


PROMPT_TEXT = """
        You are a human annotator and classify the image based on the connectedness type. Below are the definitions of different types of connectedness. Only return the result.
          • Connectedness to the self includes authenticity, inner harmony and inner peace, consciousness, self-knowledge and experiencing and searching for meaning in life.
          • Connectedness to others involves compassion, caring, gratitude and wonder.
          • Connectedness with Transcendence pertains to something or someone beyond the human level, such as the universe, transcendent reality, a higher power or God.
          • Connectedness to nature refers to the deep sense of relationship that individuals feel with the natural world and understanding of humanity’s place within the broader ecological system.
          Only return one of five labels for me: "connectedness to self", "connectedness to others", "connectedness to nature", "connectedness to Transcendence", and "hard to say".
          Image: "{image}"
          "label":
        """

SYSTEM_PROMPT = """
    You are a human annotator and classify the image based on the connectedness type. Below are the definitions of different types of connectedness. Only return the result.
    • Connectedness to the self includes authenticity, inner harmony and inner peace, consciousness, self-knowledge and experiencing and searching for meaning in life.
    • Connectedness to others involves compassion, caring, gratitude and wonder.
    • Connectedness with Transcendence pertains to something or someone beyond the human level, such as the universe, transcendent reality, a higher power or God.
    • Connectedness to nature refers to the deep sense of relationship that individuals feel with the natural world and understanding of humanity’s place within the broader ecological system.
    Only return one of five labels for me: "connectedness to self", "connectedness to others", "connectedness to nature", "connectedness to Transcendence", and "hard to say".
"""

def get_base64_encoded_image(image_path):
    with open(image_path, "rb") as image_file:
        binary_data = image_file.read()
        base_64_encoded_data = base64.b64encode(binary_data)
        base64_string = base_64_encoded_data.decode('utf-8')
        return base64_string


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def GPT4_1_annotate_image(image_path: str) -> str:

  base64_image = encode_image(image_path)

  input=[
      {"role": "system",
       "content": SYSTEM_PROMPT
      },
      {"role": "user",
       "content": [
              {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                    },
              },
          ],
      }
      ]

  response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=input,
    max_completion_tokens=300,
    temperature=0,
  )

  return response.choices[0].message.content


def GPT4o_annotate_image(image_path: str) -> str:

  base64_image = encode_image(image_path)

  input=[
      {"role": "system",
       "content": SYSTEM_PROMPT
      },
      {"role": "user",
       "content": [
              {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                    },
              },
          ],
      }
      ]

  response = openai.chat.completions.create(
    model="gpt-4o",
    messages=input,
    max_completion_tokens=300,
    temperature=0,
  )

  return response.choices[0].message.content


def GPT4omini_annotate_image(image_path: str) -> str:

  base64_image = encode_image(image_path)

  input=[
      {"role": "system",
       "content": SYSTEM_PROMPT
      },
      {"role": "user",
       "content": [
              {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                    },
              },
          ],
      }
      ]

  response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=input,
    max_completion_tokens=300,
    temperature=0,
  )

  return response.choices[0].message.content




def Claude_annotate_image(image_path: str) -> str:
    message_list = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": get_base64_encoded_image(image_path),
                    },
                },
                {
                    "type": "text",
                    "text": PROMPT_TEXT.replace("{image}", Path(image_path).name),
                },
            ],
        }
    ]

    response = claude.messages.create(
        model="claude-sonnet-4-5-20250929",
        temperature=0,
        max_tokens=256,
        messages=message_list,
        )
    content = response.content[0].text.strip()

    return content


def Gemini_annotate_image(image_path: str) -> str:
    with open(image_path, 'rb') as f:
      image_bytes = f.read()

    response = gemini.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            types.Part.from_bytes(
            data=image_bytes,
            mime_type='image/jpeg',
            ),
            SYSTEM_PROMPT
              ]
          )
    return response.text.strip()


def main():
    cols = {c.lower(): c for c in df.columns}
    path_col = cols.get("path") or "path"
    sonnet_col = "Sonnet"
    gpt4_1_col = "GPT-4.1"
    gemini_col = "Gemini"
    gpt_4o_col = "GPT-4o"
    gpt_4o_mini_col = "GPT-4o-mini"

    if path_col not in df.columns:
        raise ValueError(f"Expected a 'path' column in the input file. Found: {list(df.columns)}")

    # Annotate only missing entries, if any
    if sonnet_col not in df.columns:
        df[sonnet_col] = None

    if gpt4_1_col not in df.columns:
        df[gpt4_1_col] = None

    if gemini_col not in df.columns:
        df[gemini_col] = None

    if gpt_4o_mini_col not in df.columns:
        df[gpt_4o_mini_col] = None

    if gpt_4o_col not in df.columns:
        df[gpt_4o_col] = None

    for idx, row in df.iterrows():

        img_path = str(row[path_col])

        if not os.path.exists(img_path):
            error_msg = f"ERROR[FileNotFound]: {img_path}"
            df.at[idx, sonnet_col] = error_msg
            df.at[idx, gpt4_1_col] = error_msg
            df.at[idx, gpt_4o_col] = error_msg
            df.at[idx, gemini_col] = error_msg
            df.at[idx, gpt_4o_mini_col] = error_msg
            continue

        try:

            df.at[idx, sonnet_col] = Claude_annotate_image(img_path)

        except Exception as e:
            df.at[idx, sonnet_col] = f"ERROR[{type(e).__name__}]: {str(e).strip()}"

        try:
            df.at[idx, gpt_col] = GPT4_1_annotate_image(img_path)

        except Exception as e:
            df.at[idx, gpt_col] = f"ERROR[{type(e).__name__}]: {str(e).strip()}"

        try:
            df.at[idx, gemini_col] = Gemini_annotate_image(img_path)

        except Exception as e:
            df.at[idx, gemini_col] = f"ERROR[{type(e).__name__}]: {str(e).strip()}"

        try:

            df.at[idx, gpt_4o_mini_col] = GPT4omini_annotate_image(img_path)

        except Exception as e:
            df.at[idx, gpt_4o_mini_col] = f"ERROR[{type(e).__name__}]: {str(e).strip()}"

        try:

            df.at[idx, gpt_4o_col] = GPT4o_annotate_image(img_path)

        except Exception as e:
            df.at[idx, gpt_4o_col] = f"ERROR[{type(e).__name__}]: {str(e).strip()}"




    df.to_excel(OUTPUT_XLSX, index=False)
    print(f"Saved annotated file to: {OUTPUT_XLSX}")

if __name__ == "__main__":
    main()

Saved annotated file to: /content/drive/MyDrive/spirituality/Instagram_images_connectedness_type_goldstandard_with_paths_annotated.xlsx


**Calculate the precison, recall and F1 scores**

**Sonnet**

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# Configuration
FILE_PATH = "/content/Instagram_images_connectedness_type_goldstandard_with_paths_annotated.xlsx"
GOLD_COL = "Connectedness Type_formatted"
PRED_COL = "Sonnet_formatted"

# Load & Prepare
df = pd.read_excel(FILE_PATH)

# Only keep rows where both gold and prediction are present
pair = df.dropna(subset=[GOLD_COL, PRED_COL])[[GOLD_COL, PRED_COL]].copy()

# Normalize
pair[GOLD_COL] = pair[GOLD_COL].astype(str).str.strip().str.lower()
pair[PRED_COL] = pair[PRED_COL].astype(str).str.strip().str.lower()

y_true_all = pair[GOLD_COL].values
y_pred_all = pair[PRED_COL].values

# Determine label set (union), then remove "hard to say"
labels_all = sorted(set(y_true_all) | set(y_pred_all))
labels_no_hard = [l for l in labels_all if l != "hard to say"]

# Filter out any samples where either true or predicted is "hard to say"
mask_no_hard = (pair[GOLD_COL] != "hard to say") & (pair[PRED_COL] != "hard to say")
pair_no_hard = pair[mask_no_hard].copy()

y_true = pair_no_hard[GOLD_COL].values
y_pred = pair_no_hard[PRED_COL].values

# Metrics
# Per-class metrics for the remaining labels
p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred,
    labels=labels_no_hard,
    average=None,
    zero_division=0
)

per_class_df = pd.DataFrame({
    "label": labels_no_hard,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support (true count)": support
})

# Macro (unweighted) averages across the remaining labels
macro_precision = float(pd.Series(p).mean())
macro_recall = float(pd.Series(r).mean())
macro_f1 = float(pd.Series(f1).mean())

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Summary Table
summary_df = pd.DataFrame({
    "metric": ["macro_precision", "macro_recall", "macro_f1", "accuracy", "n_samples_used"],
    "value": [macro_precision, macro_recall, macro_f1, accuracy, len(y_true)]
})

# Output
print(summary_df)


            metric       value
0  macro_precision    0.605975
1     macro_recall    0.560276
2         macro_f1    0.571791
3         accuracy    0.739583
4   n_samples_used  192.000000


**GPT-4.1**

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# Configuration
FILE_PATH = "/content/Instagram_images_connectedness_type_goldstandard_with_paths_annotated.xlsx"
GOLD_COL = "Connectedness Type_formatted"
PRED_COL = "GPT4.1_formatted"

# Load & Prepare
df = pd.read_excel(FILE_PATH)

# Only keep rows where both gold and prediction are present
pair = df.dropna(subset=[GOLD_COL, PRED_COL])[[GOLD_COL, PRED_COL]].copy()

# Normalize
pair[GOLD_COL] = pair[GOLD_COL].astype(str).str.strip().str.lower()
pair[PRED_COL] = pair[PRED_COL].astype(str).str.strip().str.lower()

y_true_all = pair[GOLD_COL].values
y_pred_all = pair[PRED_COL].values

# Determine label set (union), then remove "hard to say"
labels_all = sorted(set(y_true_all) | set(y_pred_all))
labels_no_hard = [l for l in labels_all if l != "hard to say"]

# Filter out any samples where either true or predicted is "hard to say"
mask_no_hard = (pair[GOLD_COL] != "hard to say") & (pair[PRED_COL] != "hard to say")
pair_no_hard = pair[mask_no_hard].copy()

y_true = pair_no_hard[GOLD_COL].values
y_pred = pair_no_hard[PRED_COL].values

# Metrics
# Per-class metrics for the remaining labels
p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred,
    labels=labels_no_hard,
    average=None,
    zero_division=0
)

per_class_df = pd.DataFrame({
    "label": labels_no_hard,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support (true count)": support
})

# Macro (unweighted) averages across the remaining labels
macro_precision = float(pd.Series(p).mean())
macro_recall = float(pd.Series(r).mean())
macro_f1 = float(pd.Series(f1).mean())

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Summary Table
summary_df = pd.DataFrame({
    "metric": ["macro_precision", "macro_recall", "macro_f1", "accuracy", "n_samples_used"],
    "value": [macro_precision, macro_recall, macro_f1, accuracy, len(y_true)]
})

# Output
print(summary_df)


            metric       value
0  macro_precision    0.596807
1     macro_recall    0.484192
2         macro_f1    0.498300
3         accuracy    0.760417
4   n_samples_used  192.000000


**GPT-4o-mini**

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# Configuration
FILE_PATH = "/content/Instagram_images_connectedness_type_goldstandard_with_paths_annotated.xlsx"
GOLD_COL = "Connectedness Type_formatted"
PRED_COL = "GPT-4o-mini_formatted"

# Load & Prepare
df = pd.read_excel(FILE_PATH)

# Only keep rows where both gold and prediction are present
pair = df.dropna(subset=[GOLD_COL, PRED_COL])[[GOLD_COL, PRED_COL]].copy()

# Normalize
pair[GOLD_COL] = pair[GOLD_COL].astype(str).str.strip().str.lower()
pair[PRED_COL] = pair[PRED_COL].astype(str).str.strip().str.lower()

y_true_all = pair[GOLD_COL].values
y_pred_all = pair[PRED_COL].values

# Determine label set (union), then remove "hard to say"
labels_all = sorted(set(y_true_all) | set(y_pred_all))
labels_no_hard = [l for l in labels_all if l != "hard to say"]

# Filter out any samples where either true or predicted is "hard to say"
mask_no_hard = (pair[GOLD_COL] != "hard to say") & (pair[PRED_COL] != "hard to say")
pair_no_hard = pair[mask_no_hard].copy()

y_true = pair_no_hard[GOLD_COL].values
y_pred = pair_no_hard[PRED_COL].values

#  Metrics
# Per-class metrics for the remaining labels
p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred,
    labels=labels_no_hard,
    average=None,
    zero_division=0
)

per_class_df = pd.DataFrame({
    "label": labels_no_hard,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support (true count)": support
})

# Macro (unweighted) averages across the remaining labels
macro_precision = float(pd.Series(p).mean())
macro_recall = float(pd.Series(r).mean())
macro_f1 = float(pd.Series(f1).mean())

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Summary Table
summary_df = pd.DataFrame({
    "metric": ["macro_precision", "macro_recall", "macro_f1", "accuracy", "n_samples_used"],
    "value": [macro_precision, macro_recall, macro_f1, accuracy, len(y_true)]
})

# Output
print(summary_df)


            metric       value
0  macro_precision    0.656580
1     macro_recall    0.634201
2         macro_f1    0.639909
3         accuracy    0.758242
4   n_samples_used  182.000000


**GPT-4o**

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# -------------------- Configuration --------------------
FILE_PATH = "/content/Instagram_images_connectedness_type_goldstandard_with_paths_annotated.xlsx"
GOLD_COL = "Connectedness Type_formatted"
PRED_COL = "GPT-4o_formatted"

# Load & Prepare
df = pd.read_excel(FILE_PATH)

# Only keep rows where both gold and prediction are present
pair = df.dropna(subset=[GOLD_COL, PRED_COL])[[GOLD_COL, PRED_COL]].copy()

# Normalize
pair[GOLD_COL] = pair[GOLD_COL].astype(str).str.strip().str.lower()
pair[PRED_COL] = pair[PRED_COL].astype(str).str.strip().str.lower()

y_true_all = pair[GOLD_COL].values
y_pred_all = pair[PRED_COL].values

# Determine label set (union), then remove "hard to say"
labels_all = sorted(set(y_true_all) | set(y_pred_all))
labels_no_hard = [l for l in labels_all if l != "hard to say"]

# Filter out any samples where either true or predicted is "hard to say"
mask_no_hard = (pair[GOLD_COL] != "hard to say") & (pair[PRED_COL] != "hard to say")
pair_no_hard = pair[mask_no_hard].copy()

y_true = pair_no_hard[GOLD_COL].values
y_pred = pair_no_hard[PRED_COL].values

# Metrics
# Per-class metrics for the remaining labels
p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred,
    labels=labels_no_hard,
    average=None,
    zero_division=0
)

per_class_df = pd.DataFrame({
    "label": labels_no_hard,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support (true count)": support
})

# Macro (unweighted) averages across the remaining labels
macro_precision = float(pd.Series(p).mean())
macro_recall = float(pd.Series(r).mean())
macro_f1 = float(pd.Series(f1).mean())

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Summary Table
summary_df = pd.DataFrame({
    "metric": ["macro_precision", "macro_recall", "macro_f1", "accuracy", "n_samples_used"],
    "value": [macro_precision, macro_recall, macro_f1, accuracy, len(y_true)]
})

# Output
print(summary_df)


            metric       value
0  macro_precision    0.628723
1     macro_recall    0.604619
2         macro_f1    0.613440
3         accuracy    0.746032
4   n_samples_used  189.000000


**Gemini**

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# -------------------- Configuration --------------------
FILE_PATH = "/content/Instagram_images_connectedness_type_goldstandard_with_paths_annotated.xlsx"
GOLD_COL = "Connectedness Type_formatted"
PRED_COL = "Gemini_formatted"

# Load & Prepare
df = pd.read_excel(FILE_PATH)

# Only keep rows where both gold and prediction are present
pair = df.dropna(subset=[GOLD_COL, PRED_COL])[[GOLD_COL, PRED_COL]].copy()

# Normalize
pair[GOLD_COL] = pair[GOLD_COL].astype(str).str.strip().str.lower()
pair[PRED_COL] = pair[PRED_COL].astype(str).str.strip().str.lower()

y_true_all = pair[GOLD_COL].values
y_pred_all = pair[PRED_COL].values

# Determine label set (union), then remove "hard to say"
labels_all = sorted(set(y_true_all) | set(y_pred_all))
labels_no_hard = [l for l in labels_all if l != "hard to say"]

# Filter out any samples where either true or predicted is "hard to say"
mask_no_hard = (pair[GOLD_COL] != "hard to say") & (pair[PRED_COL] != "hard to say")
pair_no_hard = pair[mask_no_hard].copy()

y_true = pair_no_hard[GOLD_COL].values
y_pred = pair_no_hard[PRED_COL].values

# Metrics
# Per-class metrics for the remaining labels
p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred,
    labels=labels_no_hard,
    average=None,
    zero_division=0
)

per_class_df = pd.DataFrame({
    "label": labels_no_hard,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support (true count)": support
})

# Macro (unweighted) averages across the remaining labels
macro_precision = float(pd.Series(p).mean())
macro_recall = float(pd.Series(r).mean())
macro_f1 = float(pd.Series(f1).mean())

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Summary Table
summary_df = pd.DataFrame({
    "metric": ["macro_precision", "macro_recall", "macro_f1", "accuracy", "n_samples_used"],
    "value": [macro_precision, macro_recall, macro_f1, accuracy, len(y_true)]
})

# Output
print(summary_df)


            metric       value
0  macro_precision    0.637879
1     macro_recall    0.550683
2         macro_f1    0.569891
3         accuracy    0.730570
4   n_samples_used  193.000000


**LLaVA**

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# Configuration
FILE_PATH = "/content/Instagram_images_connectedness_type_goldstandard_with_paths_annotated.xlsx"
GOLD_COL = "Connectedness Type_formatted"
PRED_COL = "LLaVA_formatted"

# Load & Prepare
df = pd.read_excel(FILE_PATH)

# Only keep rows where both gold and prediction are present
pair = df.dropna(subset=[GOLD_COL, PRED_COL])[[GOLD_COL, PRED_COL]].copy()

# Normalize
pair[GOLD_COL] = pair[GOLD_COL].astype(str).str.strip().str.lower()
pair[PRED_COL] = pair[PRED_COL].astype(str).str.strip().str.lower()

y_true_all = pair[GOLD_COL].values
y_pred_all = pair[PRED_COL].values

# Determine label set (union), then remove "hard to say"
labels_all = sorted(set(y_true_all) | set(y_pred_all))
labels_no_hard = [l for l in labels_all if l != "hard to say"]

# Filter out any samples where either true or predicted is "hard to say"
mask_no_hard = (pair[GOLD_COL] != "hard to say") & (pair[PRED_COL] != "hard to say")
pair_no_hard = pair[mask_no_hard].copy()

y_true = pair_no_hard[GOLD_COL].values
y_pred = pair_no_hard[PRED_COL].values

# Metrics
# Per-class metrics for the remaining labels
p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred,
    labels=labels_no_hard,
    average=None,
    zero_division=0
)

per_class_df = pd.DataFrame({
    "label": labels_no_hard,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support (true count)": support
})

# Macro (unweighted) averages across the remaining labels
macro_precision = float(pd.Series(p).mean())
macro_recall = float(pd.Series(r).mean())
macro_f1 = float(pd.Series(f1).mean())

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Summary Table
summary_df = pd.DataFrame({
    "metric": ["macro_precision", "macro_recall", "macro_f1", "accuracy", "n_samples_used"],
    "value": [macro_precision, macro_recall, macro_f1, accuracy, len(y_true)]
})

# Output
print(summary_df)


            metric       value
0  macro_precision    0.474420
1     macro_recall    0.348322
2         macro_f1    0.389646
3         accuracy    0.403727
4   n_samples_used  161.000000


**Qwen-VL2.5**

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# Configuration
FILE_PATH = "/content/Instagram_images_connectedness_type_goldstandard_with_paths_annotated.xlsx"
GOLD_COL = "Connectedness Type_formatted"
PRED_COL = "Qwen_formatted"

# Load & Prepare
df = pd.read_excel(FILE_PATH)

# Only keep rows where both gold and prediction are present
pair = df.dropna(subset=[GOLD_COL, PRED_COL])[[GOLD_COL, PRED_COL]].copy()

# Normalize
pair[GOLD_COL] = pair[GOLD_COL].astype(str).str.strip().str.lower()
pair[PRED_COL] = pair[PRED_COL].astype(str).str.strip().str.lower()

y_true_all = pair[GOLD_COL].values
y_pred_all = pair[PRED_COL].values

# Determine label set (union), then remove "hard to say"
labels_all = sorted(set(y_true_all) | set(y_pred_all))
labels_no_hard = [l for l in labels_all if l != "hard to say"]

# Filter out any samples where either true or predicted is "hard to say"
mask_no_hard = (pair[GOLD_COL] != "hard to say") & (pair[PRED_COL] != "hard to say")
pair_no_hard = pair[mask_no_hard].copy()

y_true = pair_no_hard[GOLD_COL].values
y_pred = pair_no_hard[PRED_COL].values

# Metrics
# Per-class metrics for the remaining labels
p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred,
    labels=labels_no_hard,
    average=None,
    zero_division=0
)

per_class_df = pd.DataFrame({
    "label": labels_no_hard,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support (true count)": support
})

# Macro (unweighted) averages across the remaining labels
macro_precision = float(pd.Series(p).mean())
macro_recall = float(pd.Series(r).mean())
macro_f1 = float(pd.Series(f1).mean())

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Summary Table
summary_df = pd.DataFrame({
    "metric": ["macro_precision", "macro_recall", "macro_f1", "accuracy", "n_samples_used"],
    "value": [macro_precision, macro_recall, macro_f1, accuracy, len(y_true)]
})

# Output
print(summary_df)


            metric       value
0  macro_precision    0.557604
1     macro_recall    0.619406
2         macro_f1    0.547414
3         accuracy    0.683060
4   n_samples_used  183.000000


**Gemma3**

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# Configuration
FILE_PATH = "/content/Instagram_images_connectedness_type_goldstandard_with_paths_annotated.xlsx"
GOLD_COL = "Connectedness Type_formatted"
PRED_COL = "Gemma3_formatted"

# Load & Prepare
df = pd.read_excel(FILE_PATH)

# Only keep rows where both gold and prediction are present
pair = df.dropna(subset=[GOLD_COL, PRED_COL])[[GOLD_COL, PRED_COL]].copy()

# Normalize
pair[GOLD_COL] = pair[GOLD_COL].astype(str).str.strip().str.lower()
pair[PRED_COL] = pair[PRED_COL].astype(str).str.strip().str.lower()

y_true_all = pair[GOLD_COL].values
y_pred_all = pair[PRED_COL].values

# Determine label set (union), then remove "hard to say"
labels_all = sorted(set(y_true_all) | set(y_pred_all))
labels_no_hard = [l for l in labels_all if l != "hard to say"]

# Filter out any samples where either true or predicted is "hard to say"
mask_no_hard = (pair[GOLD_COL] != "hard to say") & (pair[PRED_COL] != "hard to say")
pair_no_hard = pair[mask_no_hard].copy()

y_true = pair_no_hard[GOLD_COL].values
y_pred = pair_no_hard[PRED_COL].values

# Metrics
# Per-class metrics for the remaining labels
p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred,
    labels=labels_no_hard,
    average=None,
    zero_division=0
)

per_class_df = pd.DataFrame({
    "label": labels_no_hard,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support (true count)": support
})

# Macro (unweighted) averages across the remaining labels
macro_precision = float(pd.Series(p).mean())
macro_recall = float(pd.Series(r).mean())
macro_f1 = float(pd.Series(f1).mean())

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Summary Table
summary_df = pd.DataFrame({
    "metric": ["macro_precision", "macro_recall", "macro_f1", "accuracy", "n_samples_used"],
    "value": [macro_precision, macro_recall, macro_f1, accuracy, len(y_true)]
})

# Output
print(summary_df)


            metric       value
0  macro_precision    0.524950
1     macro_recall    0.433494
2         macro_f1    0.348715
3         accuracy    0.450331
4   n_samples_used  151.000000
